# Workflow development

This is a little playground for me to develop functions that will be used in the final workflow.

## Imports

In [1]:
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from rdkit.Chem import Draw, PandasTools
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
import pandas as pd
import numpy as np
import MDAnalysis as mda
import nglview as nv
import os
from openmm.app import PDBFile
from pdbfixer import PDBFixer
from vina import Vina
import useful_rdkit_utils as uru
from molscrub import Scrub
from tqdm.auto import tqdm
from meeko import MoleculePreparation, PDBQTWriterLegacy, PDBQTMolecule, RDKitMolCreate

tqdm.pandas()
RDLogger.DisableLog('rdApp.*') 

## Protein preparation

**NOTE TO SELF**: Should look into replacing `os.system` with `os.subprocess` - apparently that's safer and more professional.

In [2]:
def _fix_protein(input_protein_path:str, output_directory_path:str):
    """Runs PDBFixer on the input protein structure to fix any issues such as missing residues, missing atoms, and nonstandard residues.
    The fixed structure is then saved to an internally specified output path (fixed_{input_protein_path}).

    Args:
        input_protein_path (str): The path to the input protein PDB file
        output_directory_path (str): The path to the directory where the fixed protein PDB file will be saved

    Returns:
        str: The path where the fixed protein PDB file has been saved.
    """
    # Initialise PDBFixer with input protein structure
    fixer = PDBFixer(filename=input_protein_path)

    # Identify problems with the structure and fix them
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.findNonstandardResidues()
    fixer.removeHeterogens(keepWater=False) # remove anything that isn't protein, including water and ligand(s).

    # Fix identified problems
    fixer.addMissingAtoms()
    fixer.replaceNonstandardResidues()

    # Save the fixed structure to the following output path
    output_filename = "fixed_"+os.path.split(input_protein_path)[-1] # grabs current name of input file and prepends "fixed_"
    fixed_protein_output_path = os.path.join(output_directory_path, output_filename)
    with open(fixed_protein_output_path, 'w') as f:
        # Toplology, Positions, file stream, and keep chain ID's
        PDBFile.writeFile(fixer.topology, fixer.positions, f, keepIds=True)
    
    return fixed_protein_output_path

def _protonate_protein(input_protein_path:str, output_directory_path:str, pH=7.4):
    """Runs pdb2pqr on the input protein structure to add hydrogens and assign protonated states based on the specified pH.
    The protonated structure is then saved to an internally specified output path.

    Args:
        input_protein_path (str): The path to the input protein PDB file
        output_directory_path (str): The path to the directory where the protonated protein .pqr and .pdb files will be saved
        pH (float, optional): The pH at which to assign protonated states. Defaults to 7.4 (physiological pH).

    Returns:
        str: The path where the protonated protein .pqr file has been saved.
    """

    # Defining output path and creating protonated files. A protonated .pdb file is created primarily for future visualisation.
    output_filename = os.path.split(input_protein_path)[-1].split(".")[0]+"_protonated" # grabs current name of input file, removes file extension, and appends "_protonated"
    protonated_protein_output_path = os.path.join(output_directory_path, output_filename)
    os.system(f"pdb2pqr --pdb-output={protonated_protein_output_path}.pdb --pH={pH} {input_protein_path} {protonated_protein_output_path}.pqr --whitespace")

    return f"{protonated_protein_output_path}.pqr"

def _pdb_to_pdbqt(input_protein_pqr_path:str, output_directory_path:str):
    """Writes a .pdbqt file ready for Autodock Vina given a .pqr generated by pdb2pqr.

    Args:
        input_protein_pqr_path (str): The path to the input protein .pqr file (created by pdb2pqr).
        output_directory_path (str): The path to the directory where the .pdbqt file will be saved.
    
    Returns:
    str: The path where the protein .pdbqt file has been saved.
    """

    u = mda.Universe(input_protein_pqr_path)
    output_filename = os.path.split(input_protein_pqr_path)[-1].split(".")[0]+".pdbqt" # grabs current name of input file, removes file extension, and appends ".pdbqt"
    output_filename = output_filename.replace("_protonated", "").replace("fixed", "") # removes "_protonated" and "fixed_" from the filename to match the original protein name
    pdbqt_output_path = os.path.join(output_directory_path, output_filename)
    u.atoms.write(pdbqt_output_path)

    # Read in the just-written PDBQT file, replace text, and write back
    with open(pdbqt_output_path, 'r') as file:
        file_content = file.read()

    # Replace 'TITLE' and 'CRYST1' with 'REMARK'
    file_content = file_content.replace('TITLE', 'REMARK').replace('CRYST1', 'REMARK')

    # Write the modified content back to the file
    with open(pdbqt_output_path, 'w') as file:
        file.write(file_content)
    
    return pdbqt_output_path

# Final function which will prepare the protein structure files with one function call.
def prepare_protein(input_protein_path:str, working_directory:str, pH=7.4):
    """Convenience function which fixes a protein structure with PDBFixer if any issues are present, then protonates the protein with
    pdb2pqr (propka under the hood) at a given pH, and converts the protonated structure file (.pqr used in this case) to a .pdbqt file
    ready for docking with Autodock Vina.

    Args:
        input_protein_path (str): Input protein PDB filepath.
        working_directory (str): The directory where the prepared protein files will be saved.
        pH (float, optional): The pH at which to assign protonated states. Defaults to 7.4 (physiological pH).

    Returns:
    str: The path where the prepared protein .pdbqt file has been saved.
    """
    # create protein structure output directory within the working directory
    output_directory_path = os.path.join(working_directory, "protein")
    os.makedirs(output_directory_path)

    # run protein preparation
    fixed_protein_path = _fix_protein(input_protein_path, output_directory_path)
    protonated_protein_pqr_path = _protonate_protein(fixed_protein_path, output_directory_path, pH=pH)
    pdbqt_path = _pdb_to_pdbqt(protonated_protein_pqr_path, output_directory_path)

    return pdbqt_path


## Ligand preparation

In [3]:
def _protonate_ligand(smiles:str, pH=7.4, pH_deviation=2) -> list[str]:
    """Protonate a ligand molecule given its SMILES string, using the molscrub Scrub class to assign protonation states and
    enumerate tautomers based on the specified pH and deviation range.

    Args:
        smiles (str): Ligand SMILES string.
        pH (float, optional): pH at which to protonate the ligand. Defaults to 7.4.
        pH_deviation (int, optional): Deviation from the specified pH to consider. Defaults to 2.

    Returns:
        list[str]: A list of protonated ligand SMILES strings, including tautomers.
    """

    scrub = Scrub(
    ph_low=pH - pH_deviation,
    ph_high=pH + pH_deviation,
    skip_gen3d=True # We can skip 3D generation here as we will generate 3D conformers later in the workflow after docking (using Auto3D)
    )

    mol = Chem.MolFromSmiles(smiles)
    scrubbed_mols = scrub(mol)
    protonated_smiles = [Chem.MolToSmiles(mol) for mol in scrubbed_mols]

    return protonated_smiles

# Taken and slightly edited from Auto3D stereochemistry documentation: https://auto3d.readthedocs.io/en/latest/example/stereochemistry.html
def _enumerate_stereoisomers(smiles:str, max_isomers=16, only_unassigned=True) -> list[str]:
    """Enumerate all stereoisomers of a molecule. By default, if stereochemistry is specified, it will not be enumerated
    (controlled by the only_unassigned parameter). This function treats both E/Z and R/S stereoisomerism. Generated
    stereoisomers will be checked to confirm if they can be converted into a 3D structure, and if not, it is assumed the
    molecule is nonsensical and the function will return NaN.

    Args:
        smiles (str): Input SMILES (can have undefined stereochemistry).
        max_isomers (int, optional): Maximum number of isomers to generate. Defaults to 16.
        only_unassigned (bool, optional): If set (default), stereocenters which have specified stereochemistry will not be perturbed unless they are part of a relative stereo group. Defaults to True.

    Returns:
        list[str]: List of enumerated SMILES with stereochemical information.
    """

    mol = Chem.MolFromSmiles(smiles)
    
    # Configure enumeration options
    opts = StereoEnumerationOptions(
        tryEmbedding=True,  # Verify 3D embedding is possible. If not, this function returns NaN.
        unique=True,        # Remove duplicates
        maxIsomers=max_isomers,
        onlyUnassigned=only_unassigned
    )
    
    isomers = list(EnumerateStereoisomers(mol, options=opts))
    isomer_smiles = [Chem.MolToSmiles(iso, isomericSmiles=True, canonical=True) for iso in isomers]

    return isomer_smiles

def _generate_3d_molecule(smiles:str, max_iter=1000) -> Chem.rdchem.Mol | None:
    # An attempt at generating 3D conformers of a molecule for docking without relying on auto3d.
    """Generate a 3D conformer of a given SMILES string using RDKit. The structure is then optimised using AllChem.MMFFOptimizeMolecule.

    Args:
        smiles (str): SMILES string.
        max_iter (int, optional): Maximum number of iterations to optimise a 3D structure. Defaults to 1000.

    Returns:
        Chem.rdchem.Mol | None: The 3D structure for the given SMILES, or None if generation fails.
    """
    mol = Chem.MolFromSmiles(smiles)
    mol_with_H = Chem.AddHs(mol)
    embed_status = AllChem.EmbedMolecule(mol_with_H) # Embed molecule is done inplace
    if(embed_status != 0): # if embedding fails, return None to indicate that the molecule is nonsensical and cannot be converted to 3D
        return None 
    AllChem.MMFFOptimizeMolecule(mol_with_H, maxIters=max_iter) # returns optimisation status - 0=converged, 1=not converged

    return mol_with_H

def _clean_3d_molecule_generation(mols_3d_series:pd.Series, ligand_df:pd.DataFrame, ligand_directory:str) -> tuple[pd.Series, pd.DataFrame]:
    """Cleans 3D molecule generation results by removing any molecules that failed to generate a 3D structure.
    The failed entries are saved to a CSV file for later analysis.

    Args:
        mols_3d_series (pd.Series): Series of 3D RDKit Mol objects also containing null values, as generated by _generate_3d_molecule.
        ligand_df (pd.DataFrame): DataFrame containing the ligand information corresponding to the mols_3d_series.
        ligand_directory (str): Directory path for saving failed entry information.

    Returns:
        tuple[pd.Series, pd.DataFrame]: Cleaned series of 3D RDKit Mol objects and corresponding ligand information DataFrame.
    """
    # updating mols_3d_series
    failed_to_embed_mask = mols_3d_series.isnull()
    failed_to_embed_count = mols_3d_series.isnull().sum()
    failed_to_embed_entries = ligand_df[failed_to_embed_mask]
    failed_to_embed_path = os.path.join(ligand_directory, "failed_to_embed_entries.csv")
    failed_to_embed_entries.to_csv(failed_to_embed_path, index=False) # save the failed entries to a CSV for later analysis
    print(f"{failed_to_embed_count} molecules failed to generate 3D structures. Saved failed entries to {failed_to_embed_path}.")
    mols_3d_series = mols_3d_series[~failed_to_embed_mask] # remove the failed entries from the list of 3D molecules

    # updating ligand_df to keep it consistent with the list of 3D molecules
    ligand_df = ligand_df[~failed_to_embed_mask] # remove the failed entries from the ligand dataframe as well, to keep it consistent with the list of 3D molecules
    ligand_df = ligand_df.reset_index(drop=True) # reset the index of the ligand dataframe after removing failed entries
    print(f"{ligand_df.shape[0]} molecules remain after removing failed entries.")
    return mols_3d_series, ligand_df

def _write_3d_mols_to_sdf(mol_3d_list:list[Chem.rdchem.Mol], smiles_list:list[str], sdf_path:str):
    """Write 3D molecule structures to an .sdf file for later analysis.

    Args:
        mol_3d_list (list[Chem.rdchem.Mol]): List of 3D RDKit Mol structures, as made with _generate_3d_molecule.
        smiles_list (list[str]): List of SMILES used to generate the 3D structures to save with each structure in the .sdf file.
        sdf_path (str): Output filepath for .sdf file.
    """
    writer = Chem.SDWriter(sdf_path)

    for mol, smi in zip(mol_3d_list, smiles_list):
        # Store the SMILES as an SDF property
        mol.SetProp("_SMILES", smi)
        writer.write(mol)
    
    writer.close()

def _write_ligand_pdbqt(mol_3d:Chem.rdchem.Mol, pdbqt_path:str):
    """Write a .pdbqt file from a 3D RDKit Mol object, as generated by _generate_3d_molecule.

    Args:
        mol_3d (Chem.rdchem.Mol): 3D RDKit Mol object, as generated by _generate_3d_molecule.
        pdbqt_path (str): Output filepath for .pdbqt file.
    """
    meeko_prep = MoleculePreparation()
    mol_setup = meeko_prep.prepare(mol_3d)[0] # there will only ever be one molecule setup in this basic workflow
    pdbqt_string = PDBQTWriterLegacy.write_string(mol_setup)[0] # write_string returns a tuple - the pdbqt string and an error message (if relevant, otherwise it's blank)

    # file writing
    with open(pdbqt_path, "w") as f:
        f.write(pdbqt_string)

def _write_multiple_ligand_pdbqt_files(mol_3d_list:list[Chem.rdchem.Mol], pdbqt_output_directory:str) -> str:
    """A version of _write_ligand_pdbqt which applies to a list of 3D RDKit Mol objects to write .pdbqt files
    for each molecule in the list. This function returns a list of .pdbqt filepaths for each molecule passed in.

    Args:
        mol_3d_list (list[Chem.rdchem.Mol]): 3D RDKit Mol objects, as generated by _generate_3d_molecule.
        pdbqt_output_directory (str): Output directory for .pdbqt files.

    Returns:
        list[str]: List of .pdbqt filepaths for each molecule passed in.
    """
    ligand_pdbqt_dir = os.path.join(pdbqt_output_directory, "ligand_pdbqts")
    os.makedirs(ligand_pdbqt_dir)
    ligand_pdbqt_filepath_list = []
    for i, mol in enumerate(tqdm(mol_3d_list)):
        ligand_pdbqt_filepath = os.path.join(ligand_pdbqt_dir, f"{i}.pdbqt")
        ligand_pdbqt_filepath_list.append(ligand_pdbqt_filepath)
        _write_ligand_pdbqt(mol, ligand_pdbqt_filepath)
    return ligand_pdbqt_filepath_list

def prepare_ligands(smiles_list:list[str], generation_round_dir:str, pH=7.4, pH_deviation=2, enumerate_only_unassigned=True, max_iter=1000) -> list[str]:
    """Convenience function which generates the protonation states and tautomers of a list of SMILES, then enumerates
    stereoisomers, generating 3D structures for each stereoisomer and optimising the structures, and saving .pdbqt files
    for docking with Autodock Vina.

    Args:
        smiles_list (list[str]): List of SMILES strings.
        generation_round_dir (str): Directory for the current generation round.
        pH (float, optional): pH to protonate ligands. Defaults to 7.4.
        pH_deviation (int, optional): Deviation from the specified pH to consider for ligand protonation. Defaults to 2.
        enumerate_only_unassigned (bool, optional): If set (default), stereocenters which have specified stereochemistry will not be perturbed unless they are part of a relative stereo group. Defaults to True.
        max_iter (int, optional): Maximum number of iterations to optimise a 3D structure. Defaults to 1000.
    
    Returns:
        tuple[list[str], str]: List of absolute filepaths for each generated ligand .pdbqt file and the path to the enumerated ligands CSV file.
    """
    ligand_directory = os.path.join(generation_round_dir, "ligand_prep")
    os.makedirs(ligand_directory)

    # For below: progress_apply is from tqdm.pandas() and applies a function whilst showing a progress bar
    ligand_df = pd.DataFrame({"annalog": smiles_list})

    print("Protonating molecules...")
    ligand_df["protonated"] = ligand_df["annalog"].progress_apply(_protonate_ligand, pH=pH, pH_deviation=pH_deviation)
    ligand_df = ligand_df.explode("protonated", ignore_index=True)

    print("Enumerating isomers...")
    ligand_df["isomers"] = ligand_df["protonated"].progress_apply(_enumerate_stereoisomers, only_unassigned=enumerate_only_unassigned)
    ligand_df = ligand_df.explode("isomers", ignore_index=True)
    invalid_smiles = ligand_df.isnull().sum().sum() # when generating stereoisomers, NaN can be returned if a SMILES cannot be converted to 3D, thus it is considered nonsensical.
    print(f"{invalid_smiles} invalid SMILES detected.")
    if(invalid_smiles > 0):
        ligand_df = ligand_df.dropna()
        print("Invalid SMILES removed.")

    print("Deduplicating isomers...")
    len_before_deduplication = ligand_df.shape[0]
    ligand_df = ligand_df.drop_duplicates("isomers", ignore_index=True)
    len_after_deduplication = ligand_df.shape[0]
    print(f"{len_before_deduplication} molecules identified after enumeration.", end=" ")
    print(f"{len_before_deduplication-len_after_deduplication} entries removed.")

    print("Generating 3D structures per SMILES...") # this is done more to check the structures later on, if the user wishes
    mols_3d = ligand_df["isomers"].progress_apply(_generate_3d_molecule, max_iter=max_iter)
    if(mols_3d.isnull().any()): # it is possible for 3D generation to fail despite the check during stereo enumeration.
        mols_3d, ligand_df = _clean_3d_molecule_generation(mols_3d, ligand_df, ligand_directory)
    mols_3d = mols_3d.to_list() # convert to list for later use in writing .pdbqt and .sdf files

    sdf_path = os.path.join(ligand_directory, "prepped_ligands.sdf")
    print(f"Saving 3D structures to {sdf_path}")
    _write_3d_mols_to_sdf(mols_3d, ligand_df["isomers"].to_list(), sdf_path)

    ligand_info_path = os.path.join(ligand_directory, "prepped_ligands.csv")
    print(f"Saving enumerated ligand smiles to {ligand_info_path}")
    ligand_df.insert(0, "sdf_number", ligand_df.index+1) # Add a column for the sdf number, which is the index of the molecule in the .sdf file (1-indexed)
    ligand_df.to_csv(ligand_info_path, index=False)
    
    print("Writing .pdbqt files of each ligand...")
    ligand_pdbqt_filepath_list = _write_multiple_ligand_pdbqt_files(mols_3d, ligand_directory)
    print(f"{len(ligand_pdbqt_filepath_list)} molecules have been prepped for docking.")
    
    return ligand_pdbqt_filepath_list, ligand_info_path

## ANNalog generation setup

In [4]:
def _annalog_generate(n_mol_to_gen:int, input_path_or_smiles:str, output_directory:str, method="beam", number_of_variants=5) -> str:
    """Generate SMILES with ANNalog. Variants are used to generate multiple variant SMILES per input SMILES.
    The total number of molecules to generate is passed in, and internally this is converted into a number of molecules
    to generate per input SMILES, additionally given the number of variants to generate per input SMILES.

    Args:
        n_mol_to_gen (int): Number of molecules to generate in total.
        input_path_or_smiles (str): Path to input file or SMILES string
        output_directory (str): Path to output directory.
        method (str, optional): Generation method. Defaults to "beam". Other options are: "BF-beam" and "sampling".
        number_of_variants (int, optional): Number of variants to generate. Defaults to 5.
    
    Returns:
        str: Path to the .tsv file containing generated molecules.
    """
    # computing the number of molecules to generate per input SMILES, given the total number of molecules to generate.
    n_mol_to_gen_per_input = n_mol_to_gen // number_of_variants # integer division to ensure we get an integer number of molecules to generate per input SMILES.
    filename = "annalog_generated_molecules.tsv"
    output_filepath = os.path.join(output_directory, filename)
    os.system(f"annalog-generate -i '{input_path_or_smiles}' -n {n_mol_to_gen_per_input} -m {method} -o {output_filepath} -e variants --variant-number {number_of_variants}")
    return output_filepath

def _load_generated_molecules(generated_molecules_path:str):
    """Convenience function to load generated molecules from a .tsv file and add RDKit molecule columns for both input and generated SMILES.

    Args:
        generated_molecules_path (str): Path to the .tsv file containing generated molecules.

    Returns:
        pd.DataFrame: DataFrame containing the generated molecules with RDKit molecule columns.
    """

    df = pd.read_csv(generated_molecules_path, sep="\t")
    return df

def _deduplicate_generated_molecules(generated_df:pd.DataFrame) -> list[str]:
    """Deduplicates a DataFrame created from ANNalog's generation output. Duplicated generated molecules are removed, along with generated
    molecules that are identical to an input molecule.

    Args:
        generated_df (pd.DataFrame): DataFrame based on ANNalog generation output.

    Returns:
        list[str]: List of deduplicated generated SMILES strings.
    """

    generated_df["canonical_input_smiles"] = generated_df["input_smiles"].apply(Chem.CanonSmiles)
    generated_df["canonical_generated_smiles"] = generated_df["generated_smiles"].apply(Chem.CanonSmiles)
    generated_df = generated_df.drop_duplicates("canonical_generated_smiles") # remove duplicate generated molecules
    generated_df = generated_df[~generated_df["canonical_generated_smiles"].isin(generated_df["canonical_input_smiles"])] # remove generated molecules that are the same as its input molecule
    generated_df = generated_df.reset_index(drop=True)
    return generated_df

def is_smiles(string:str) -> bool:
    """Check if a string is a valid SMILES string.

    Args:
        string (str): String to check.

    Returns:
        bool: True if the string is a valid SMILES, False otherwise.
    """
    try:
        mol = Chem.MolFromSmiles(string)
        return mol is not None
    except Exception:
        return False

def generate_and_deduplicate(n_mol_to_gen:int, input_path_or_smiles:str, generation_round_dir:str, method="beam", number_of_variants=5, internal_multiplier:float=2):
    """Convenience function which generates molecules with ANNalog and deduplicates the results. Internally, n_mol_to_gen
    is multiplied by internal_multiplier to account for duplicates, which will be removed later. Excess molecules, above
    the original n_mol_to_gen, will be discarded after deduplication, keeping molecules with higher rank as assigned by ANNalog.

    Args:
        n_mol_to_gen (int): Number of molecules to generate in total.
        input_path_or_smiles (str): Path to input file or SMILES string
        generation_round_dir (str): Directory for the current generation round.
        method (str, optional): Generation method. Defaults to "beam". Other options are: "BF-beam" and "sampling".
        number_of_variants (int, optional): Number of variants to generate. Defaults to 5.
        internal_multiplier (float, optional): Multiplier for the internal number of molecules to generate. Defaults to 2.

    Returns:
        list[str]: List of deduplicated generated SMILES strings.
    """
    if(not is_smiles(input_path_or_smiles)): # recalculating n_mol_to_gen if input is a file of multiple SMILES
        with open(input_path_or_smiles, 'r') as fp:
            num_of_input_mols = sum(1 for line in fp)
        n_mol_to_gen //= num_of_input_mols # divide the number of molecules to generate by the number of input molecules to get the number of molecules to generate per input molecule
    else:
        num_of_input_mols = 1 # if the input is a single SMILES, then the number of input molecules is 1

    internal_n_mol_to_gen = int(internal_multiplier * n_mol_to_gen) # increase the number of molecules to generate to account for duplicates, which will be removed later
    print(f"Internally generating {internal_n_mol_to_gen} molecules per input molecule ({num_of_input_mols} molecule(s)) to account for duplicates.")
    output_file = _annalog_generate(internal_n_mol_to_gen, input_path_or_smiles, generation_round_dir, method=method, number_of_variants=number_of_variants)
    generated_df = _load_generated_molecules(output_file)
    generated_df = _deduplicate_generated_molecules(generated_df)
    print(f"Deduplicated generated molecules. {generated_df.shape[0]} molecules remain after deduplication.\nNow trimming to maximum of {n_mol_to_gen} molecules per input molecule.")
    # take only the first n_mol_to_gen generated molecules per input molecule after deduplication - molecules are already ordered by rank
    generated_df = generated_df.groupby("input_smiles").head(n_mol_to_gen)
    print("Number of generated molecules per input molecule after trimming:")
    print(generated_df["input_smiles"].value_counts())
    generated_smiles = generated_df["generated_smiles"].to_list() # return only a list of generated, deduplicated SMILES

    return generated_smiles

## Docking functions

### Setup

In [5]:
def _get_receptor_grid(protein_with_ligand_struct_path:str, buffer=5):
    """Calculate a docking receptor grid's box centre and box dimensions based on a protein-ligand co-crystal.
    The box centre is calculated as the centroid of the bound ligand, and the box dimensions are calculated as
    the maximum extent of the ligand in each dimension, plus a buffer to ensure the ligand fits comfortably within
    the box.

    Args:
        protein_with_ligand_struct_path (str): Filepath to a protein-ligand co-crystal structure (PDB format).
        buffer (int, optional): Buffer size in Angstroms to add to the box dimensions. Defaults to 5.

    Returns:
        tuple[list]: A tuple containing the box centre and box dimensions.
    """
    u = mda.Universe(protein_with_ligand_struct_path)
    ligand = u.select_atoms("not protein")
    box_centre = ligand.center_of_geometry()
    box_centre = box_centre.tolist()
    box_dimensions = np.ceil(ligand.positions.max(axis=0) - ligand.positions.min(axis=0)) + buffer # np.ceil is used to round up the dimensions to the larger nearest integer for ease of reporting size.
    box_dimensions = box_dimensions.tolist()
    print(f"Receptor grid centre: {box_centre}\nReceptor grid dimensions: {box_dimensions}")

    return box_centre, box_dimensions

def vina_setup_receptor(vina_object:Vina, protein_with_ligand_struct_path:str, protein_pdbqt_path:str, grid_buffer:float=5) -> Vina:
    """Initialise and setup a Vina object with a receptor structure and receptor grid.

    Args:
        vina_object (Vina): An instance of the Vina class from the vina package.
        protein_with_ligand_struct_path (str): Filepath to a protein-ligand co-crystal structure (PDB format) to setup receptor grid.
        protein_pdbqt_path (str): Filepath to the prepared protein .pdbqt file.
        grid_buffer (float, optional): Buffer size in Angstroms to add to the receptor grid dimensions around cognate ligand. Defaults to 5.

    Returns:
        Vina: The initialised Vina object with the receptor structure and receptor grid set up.
    """
    box_centre, box_dimensions = _get_receptor_grid(protein_with_ligand_struct_path, buffer=grid_buffer)
    vina_object.set_receptor(protein_pdbqt_path)
    vina_object.compute_vina_maps(center=box_centre, box_size=box_dimensions)
    return vina_object

def _vina_dock_ligand(vina_object:Vina, ligand_pdbqt:str, n_poses:int, exhaustiveness:int) -> Vina:
    """Dock a ligand to a receptor using a Vina object already set up with vina_setup_receptor.

    Args:
        vina_object (Vina): An instance of the Vina class from the vina package, with the receptor structure and grid set up.
        ligand_pdbqt (str): Filepath to the prepared ligand .pdbqt file.
        n_poses (int): Number of docking poses to generate.
        exhaustiveness (int): Exhaustiveness of the global search. Higher values lead to more thorough searches but take longer.

    Returns:
        list[tuple]: A list of tuples containing the docking poses and their corresponding scores.
    """
    vina_object.set_ligand_from_file(ligand_pdbqt)
    vina_object.dock(exhaustiveness=exhaustiveness, n_poses=n_poses)
    return vina_object

### Pose and energy collection

In [6]:
def _get_best_docking_energy(vina_object:Vina) -> float:
    """Returns the docking score (energy) of the best docked pose from Autodock Vina.

    Args:
        vina_object (Vina): Vina object after docking has been performed to take the energy from.

    Returns:
        float: The best docked pose energy.
    """
    best_energy = vina_object.energies()[0,0] # this grabs the total energy of the best (lowest energy) pose
    return best_energy

def _save_docking_energies(smiles_df_path:str, output_path:str, docking_energy_list:list[float]):
    """Given a list of energies as acquired through get_best_docking_energy, save the results to the (likely .csv) file
    generated earlier in the workflow that contains the SMILES strings of the different isomers generated from the
    original ANNalog SMILES.

    Args:
        smiles_df_path (str): Path to file containing SMILES used to generate 3D molecule structures used for docking.
        output_path (str): Path to save the modified file with docking scores/energies.
        docking_energy_list (list[float]): List of docking score energies per molecule docked.
    """
    smiles_df = pd.read_csv(smiles_df_path)
    smiles_df["docking_energy"] = docking_energy_list
    smiles_df.to_csv(output_path, index=False)

def _get_best_docking_pose(vina_object:Vina) -> str:
    """Returns the best docked pose for a molecule from Autodock Vina.

    Args:
        vina_object (Vina): Vina object after docking has been performed to take the pose from.

    Returns:
        str: PDBQT file string of the docked pose.
    """
    output_pdbqt = vina_object.poses(n_poses=1)
    return output_pdbqt

def _save_docking_poses(pose_pdbqt_list:list[str], sdf_output_path:str):
    """Save the docking poses collected from each molecule into one .sdf file.

    Args:
        pose_pdbqt_list (list[str]): List of PDBQT strings as generated by get_best_docking_pose.
        smiles_list (list[str]): List of SMILES used to generated the molecules that were docked, used to assign bond orders before saving.
        sdf_output_path (str): Filepath to save .sdf file to.
    """
    f = Chem.SDWriter(sdf_output_path)
    for pdbqt in pose_pdbqt_list:
        pmol = PDBQTMolecule(pdbqt, skip_typing=True)
        rdmol_list = RDKitMolCreate.from_pdbqt_mol(pmol)
        best_pose = rdmol_list[0] # there should only be one pose saved. Otherwise one should iterate over pmol to save each pose.
        f.write(best_pose)
    f.close()

### Full docking protocol function

In [7]:
def _dock_and_score(ligand_pdbqt_filepath_list:list[str], prepared_vina_object:Vina, n_poses:int, exhaustiveness:int) -> tuple[list[float], list[str]]:
    """Given a list of prepared ligand PDBQT filepaths, dock to a receptor as defined in a prepared Vina object and return
    a list of docking scores (energies) and pose PDBQT strings per ligand.

    Args:
        ligand_pdbqt_filepath_list (list[str]): List of prepared ligand PDBQT filepaths.
        prepared_vina_object (Vina): A set up Vina object that already has a receptor and receptor grid set up, as done with vina_setup_receptor.
        n_poses (int): Vina parameter - Number of poses to generate for docking. This controls how many poses are returned after docking.
        exhaustiveness (int): Vina parameter - Exhaustiveness of the global search. Higher values lead to more thorough searches but take longer.

    Returns:
        tuple[list[float], list[str]]: A tuple of, 1. a list of docking scores per ligand, and 2. a list of pose PDBQT strings per ligand.
    """
    
    docking_energy_list = []
    docked_pose_pdbqt_list = []
    for ligand in tqdm(ligand_pdbqt_filepath_list):
        v = _vina_dock_ligand(prepared_vina_object, ligand, n_poses=n_poses, exhaustiveness=exhaustiveness)
        docking_energy_list.append(_get_best_docking_energy(v))
        docked_pose_pdbqt_list.append(_get_best_docking_pose(v))
    
    return docking_energy_list, docked_pose_pdbqt_list

def _save_docking_results(smiles_df_path:str, output_directory:str, docking_energy_list:list[float], docking_pose_pdbqt_list:list[str]) -> str:
    """Save docking poses as a multi-ligand .sdf and energies as a .csv file associated with ligand SMILES strings.

    Args:
        smiles_df_path (str): Filepath to file containing enumerated isomer SMILES made earlier in the workflow.
        output_directory (str): Directory to save docking results.
        docking_energy_list (list[float]): List of docking scores as generated with dock_and_score.
        docking_pose_pdbqt_list (list[str]): List of docked pose PDBQT strings as generated with dock_and_score.
    
    Returns:
        str: Filepath to the saved .csv file containing SMILES and docking energies.
    """
    
    docking_results_output_path = os.path.join(output_directory, "smiles_and_energies.csv")
    pose_sdf_path = os.path.join(output_directory, "docked_poses.sdf")
    _save_docking_energies(smiles_df_path, docking_results_output_path, docking_energy_list)
    print(f"Energies saved to {docking_results_output_path}")
    _save_docking_poses(docking_pose_pdbqt_list, pose_sdf_path)
    print(f"Poses saved to {pose_sdf_path}")

    return docking_results_output_path

def docking(ligand_pdbqt_filepath_list:list[str], prepared_vina_object:Vina, n_poses:int, exhaustiveness:int, smiles_df_path:str, generation_round_dir:str):
    """Convenience function to perform docking and save docking results.

    Args:
        ligand_pdbqt_filepath_list (list[str]): List of prepared ligand PDBQT filepaths.
        prepared_vina_object (Vina): A set up Vina object that already has a receptor and receptor grid set up, as done with vina_setup_receptor.
        n_poses (int): Vina parameter - Number of poses to generate for docking. This controls how many poses are returned after docking.
        exhaustiveness (int): Vina parameter - Exhaustiveness of the global search. Higher values lead to more thorough searches but take longer.
        smiles_df_path (str): Filepath to file containing enumerated isomer SMILES made earlier in the workflow.
        generation_round_dir (str): Directory for the current generation round.
    
    Returns:
        str: Filepath to the saved .csv file containing SMILES and docking energies.
    """
    docking_results_directory = os.path.join(generation_round_dir, "docking")
    os.makedirs(docking_results_directory)

    docking_energy_list, docked_pose_pdbqt_list = _dock_and_score(ligand_pdbqt_filepath_list, prepared_vina_object, n_poses, exhaustiveness)
    docking_results_output_path = _save_docking_results(smiles_df_path, docking_results_directory, docking_energy_list, docked_pose_pdbqt_list)

    return docking_results_output_path

## Clustering and candidate selection

In [8]:
def _cluster(results_filepath:str, output_directory:str, clustering_threshold=0.4) -> pd.DataFrame:
    """Cluster the docking results based on the isomer SMILES strings using Butina clustering (with Morgan2 fingerprints of length 1024),
    and save the results with cluster labels to a new file. Additionally, return the results with cluster labels as a DataFrame for subsequent candidate selection.

    Args:
        results_filepath (str): Path to the CSV file containing docking results.
        output_directory (str): Path to the directory where a CSV file with the clustered results will be saved.
        clustering_threshold (float, optional): Tanimoto distance threshold for Butina clustering. Defaults to 0.4.

    Returns:
        pd.DataFrame: The DataFrame with clustered results.
    """
    results_df = pd.read_csv(results_filepath, index_col=0)
    clustering = uru.get_butina_clusters(results_df["isomers"], clustering_threshold)

    results_df_with_clusters = results_df.copy()
    results_df_with_clusters["cluster"] = clustering

    output_file = os.path.join(output_directory, "clustered_results.csv")
    results_df_with_clusters.to_csv(output_file) # save the results with clusters to a new file for future analysis

    return results_df_with_clusters # additionally return the results with clusters as a DataFrame for candidate selection

def _select_candidates_docking_score(results_df_with_clusters:pd.DataFrame, threshold_buffer=2, candidates_to_save=5) -> pd.DataFrame:
    """Select generation candidates for the next generation-docking round based on docking scores within a threshold (2 kcal/mol above the best score by default).
    Each candidate comes from a different cluster, and the best scoring candidate per cluster is chosen as the cluster representative.

    Args:
        results_df_with_clusters (pd.DataFrame): DataFrame with clustered results.
        threshold_buffer (int, optional): Buffer for docking score thresholding - the final threshold is set to the best docking score plus this buffer. Defaults to 2.
        candidates_to_save (int, optional): Number of candidates to save total, where each candidate comes from a different cluster. Defaults to 5.

    Returns:
        pd.DataFrame: DataFrame containing the selected candidates.
    """
    best_docking_score = results_df_with_clusters["docking_energy"].min()
    docking_score_threshold = best_docking_score+threshold_buffer
    candidates_below_score_threshold_mask = results_df_with_clusters["docking_energy"]<docking_score_threshold # boolean mask with candidates below threshold
    results_df_with_clusters_filtered_by_threshold = results_df_with_clusters[candidates_below_score_threshold_mask] # filter by threshold
    best_candidates_by_cluster_and_score = (results_df_with_clusters_filtered_by_threshold
                                            .sort_values(["cluster", "docking_energy"]) # sort first by cluster, and then by docking energy
                                            .drop_duplicates("cluster")                 # only keep lowest docking score ligand per cluster
                                            .sort_values("docking_energy")              # sort by docking energy to have best scoring candidates first
                                            .head(candidates_to_save)[["isomers","docking_energy"]]) # select top candidates_to_save candidates, saving only their SMILES and docking score
    best_candidates_by_cluster_and_score = best_candidates_by_cluster_and_score.rename(columns={"isomers": "smiles"})
    return best_candidates_by_cluster_and_score

def _select_candidates_random(results_df_with_clusters:pd.DataFrame, candidates_to_save=5):
    """Select generation candidates for the next generation-docking round randomly, where each candidate comes from a different cluster.

    Args:
        results_df_with_clusters (pd.DataFrame): DataFrame with clustered results.
        candidates_to_save (int, optional): Number of candidates to save total, where each candidate comes from a different cluster. Defaults to 5.

    Returns:
        pd.DataFrame: DataFrame containing the selected candidates.
    """
    random_candidates_by_cluster = (results_df_with_clusters
                                    .groupby("cluster") # group by cluster
                                    .sample(1)          # select one candidate per cluster to act as cluster representatives
                                    .sample(candidates_to_save)) # randomly select candidates_to_save cluster representatives
    try:
        random_candidates_by_cluster = random_candidates_by_cluster[["isomers", "docking_energy"]] # if docking energies are present, save them
    except:
        random_candidates_by_cluster = random_candidates_by_cluster["isomers"]

    random_candidates_by_cluster = random_candidates_by_cluster.rename(columns={"isomers": "smiles"})
    return random_candidates_by_cluster

def cluster_and_select_candidates(results_filepath:str, generation_round_dir:str, clustering_threshold=0.4, selection_method="docking_score", threshold_buffer=2, candidates_to_save=5):
    """Cluster the docking results and select candidates based on the specified selection method. Clustered results are saved to a CSV file,
    along with the selected candidates for the next generation-docking round. The selection method can be based on docking scores or random
    selection.

    Args:
        results_filepath (str): Path to the CSV file containing docking results.
        generation_round_dir (str): Directory for the current generation round.
        clustering_threshold (float, optional): Tanimoto distance threshold for Butina clustering. Defaults to 0.4.
        selection_method (str, optional): Method for selecting candidates. Options are "docking_score" or "random". Defaults to "docking_score".
        threshold_buffer (int, optional): Buffer for docking score thresholding when using "docking_score" selection method (final threshold is set to the best docking score plus this buffer). Defaults to 2.
        candidates_to_save (int, optional): Number of candidates to save total, where each candidate comes from a different cluster. Defaults to 5.

    Returns:
        str: Path to the CSV file containing the selected candidates.
    """
    clustering_results_directory = os.path.join(generation_round_dir, "clustering")
    os.makedirs(clustering_results_directory)

    results_df_with_clusters = _cluster(results_filepath, clustering_results_directory, clustering_threshold)

    if selection_method == "docking_score":
        selected_candidates = _select_candidates_docking_score(results_df_with_clusters, threshold_buffer, candidates_to_save)
    elif selection_method == "random":
        selected_candidates = _select_candidates_random(results_df_with_clusters, candidates_to_save)
    else:
        raise ValueError("Invalid selection method. Choose 'docking_score' or 'random'.")

    selected_candidates_filepath = os.path.join(clustering_results_directory, "generation_candidates.csv")
    selected_candidates.to_csv(selected_candidates_filepath, index=False)
    return selected_candidates_filepath

def get_generation_candidates_smi(generation_candidates_filepath:str, generation_round_dir:str) -> str:
    """Convert the previous generation-docking round's generation candidates file into a .smi file for the current
    generation-docking round. The filepath of the generation candidates .smi file is returned to be passed into
    ANNalog.

    Args:
        generation_candidates_filepath (str): Path to the CSV file containing generation candidates acquired at the end of the previous round.
        generation_round_dir (str): Directory for the current generation round.
    Returns:
        str: Path to the generated .smi file containing generation candidates.
    """

    gen_candidates_list = pd.read_csv(generation_candidates_filepath)["smiles"].to_numpy()
    gen_candidates_smi_filepath = os.path.join(generation_round_dir, "generation_candidates.smi")
    np.savetxt(gen_candidates_smi_filepath, gen_candidates_list, fmt="%s") # save the generation candidates to a text file for use in the next round of generation and docking
    return gen_candidates_smi_filepath

# Workflow

In [9]:
# workflow settings
workflow_rounds = 2 # run the workflow workflow_rounds times

## preparation settings (protein+ligand)
input_smiles = "Cc1c(n(nc1C(=O)NC23CC4CC(C2)CC(C4)C3)CCCCCO)c5ccccc5" # PDB ligand ID: 9JU, bound to PDB ID 5ZTY
pH = 7.4 # physiological pH - specified for ligand and protein protonation
pH_deviation = 2 # deviation of +/- 2 log units for ligand protonation
enumerate_only_unassigned = True # only enumerate undefined stereocentres.
max_opt_iter = 1000 # when generating 3D molecules, this is the maximum number of optimisation steps for the 3D geometry optimisation

## docking settings
exhaustiveness = 16 # doubled from the default of 8
n_poses = 1 # only save the best pose per ligand.
grid_buffer = 5 # buffer in Angstroms to add to the receptor grid dimensions around the co-crystal input structure cognate ligand.


## generation settings
num_of_mols_to_gen = 100
gen_algo = "beam"
internal_multipier = 2 # internally, generate 2 times the number of molecules to account for duplicates, which will be removed later

# input and output directories for the workflow
workflow_results_dir = os.path.join("results", "workflow") # working directory for the workflow results, which will contain subdirectories for each round of generation and docking.
run_directory = os.path.join(workflow_results_dir, gen_algo) # naming the directory to save results to after the generation algorithm used
input_protein_structure_path = os.path.join("Structures", "refined_cnr2_human_5ZTY_inactive.pdb") # protein structure to dock to (refined version of PDB ID: 5ZTY from GPCRdb - CB2 receptor)

# protein preparation
print("Preparing protein structure for docking...")
protein_pdbqt_path = prepare_protein(input_protein_structure_path, workflow_results_dir, pH=pH)

v=Vina(sf_name="vina", verbosity=0) # using the default vina scoring function, and setting the verbosity to 0 to silence Vina output
v=vina_setup_receptor(v, input_protein_structure_path, protein_pdbqt_path, grid_buffer=grid_buffer)

Preparing protein structure for docking...


INFO:PDB2PQR v3.7.1: biomolecular structure conversion software.
INFO:Please cite:  Jurrus E, et al.  Improvements to the APBS biomolecular solvation software suite.  Protein Sci 27 112-128 (2018).
INFO:Please cite:  Dolinsky TJ, et al.  PDB2PQR: expanding and upgrading automated preparation of biomolecular structures for molecular simulations. Nucleic Acids Res 35 W522-W525 (2007).
INFO:Checking and transforming input arguments.
INFO:Loading topology files.
INFO:Loading molecule: results/workflow/protein/fixed_refined_cnr2_human_5ZTY_inactive.pdb
INFO:Setting up molecule.
INFO:Created biomolecule object with 360 residues and 2785 atoms.
INFO:Setting termini states for biomolecule chains.
INFO:Loading forcefield.
INFO:Loading hydrogen topology definitions.
INFO:This biomolecule is clean.  No repair needed.
INFO:Updating disulfide bridges.
INFO:Debumping biomolecule.
INFO:Adding hydrogens to biomolecule.
INFO:Debumping biomolecule (again).
INFO:Optimizing hydrogen bonds
INFO:Applying fo

Receptor grid centre: [9.093451592230029, -0.16790321373170422, -55.723128780241936]
Receptor grid dimensions: [13.0, 16.0, 15.0]


In [10]:
selected_candidates_filepath = None # initialise the selected candidates filepath to None, which will be updated after each round of generation and docking.

for round_num in range(0, workflow_rounds):
    print(f"Starting round {round_num} of {workflow_rounds}...")
    print("-"*60)

    generation_round_dir = os.path.join(run_directory, f"round_{round_num}")
    os.makedirs(generation_round_dir)

    if(round_num != 0): # for subsequent rounds, the input SMILES is the generation candidates from the previous round
        input_smiles = get_generation_candidates_smi(selected_candidates_filepath, generation_round_dir)

    print(f"Generating {num_of_mols_to_gen} molecules...")
    generated_smiles = generate_and_deduplicate(num_of_mols_to_gen, input_smiles, generation_round_dir, method=gen_algo, internal_multiplier=internal_multipier)
    print(f"{len(generated_smiles)}/{num_of_mols_to_gen} molecules remain after deduplication.")
    print("-"*60)

    print("Preparing ligands for docking...")
    ligand_pdbqt_filepaths, enumerated_smiles_path = prepare_ligands(generated_smiles,
                                                                    generation_round_dir,
                                                                    pH=pH,
                                                                    pH_deviation=pH_deviation,
                                                                    enumerate_only_unassigned=enumerate_only_unassigned,
                                                                    max_iter=max_opt_iter)


    print("Docking...")
    docking_results_filepath = docking(ligand_pdbqt_filepaths, v, n_poses, exhaustiveness, enumerated_smiles_path, generation_round_dir)

    selected_candidates_filepath = cluster_and_select_candidates(docking_results_filepath, generation_round_dir, selection_method="docking_score")


Starting round 0 of 2...
------------------------------------------------------------
Generating 100 molecules...
Internally generating 200 molecules per input molecule (1 molecule(s)) to account for duplicates.


[INFO] input SMILES changed after RDKit normalization, using RDKit processed version : Cc1c(-c2ccccc2)n(CCCCCO)nc1C(=O)NC12CC3CC(C1)CC(C3)C2 as input


Deduplicated generated molecules. 109 molecules remain after deduplication.
Now trimming to maximum of 100 molecules per input molecule.
Number of generated molecules per input molecule after trimming:
input_smiles
Cc1c(n(nc1C(=O)NC23CC4CC(C2)CC(C4)C3)CCCCCO)c5ccccc5    100
Name: count, dtype: int64
100/100 molecules remain after deduplication.
------------------------------------------------------------
Preparing ligands for docking...
Protonating molecules...


  0%|          | 0/100 [00:00<?, ?it/s]

Enumerating isomers...


...

1 molecules failed to generate 3D structures. Saved failed entries to results/workflow/beam/round_1/ligand_prep/failed_to_embed_entries.csv.
96 molecules remain after removing failed entries.
Saving 3D structures to results/workflow/beam/round_1/ligand_prep/prepped_ligands.sdf
Saving enumerated ligand smiles to results/workflow/beam/round_1/ligand_prep/prepped_ligands.csv
Writing .pdbqt files of each ligand...


  0%|          | 0/96 [00:00<?, ?it/s]

96 molecules have been prepped for docking.
Docking...


  0%|          | 0/96 [00:00<?, ?it/s]

Energies saved to results/workflow/beam/round_1/docking/smiles_and_energies.csv
Poses saved to results/workflow/beam/round_1/docking/docked_poses.sdf
